In [9]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint
from typing import TypedDict
from dotenv import load_dotenv

In [6]:
class BatsmanState(TypedDict):
    runs: int
    balls: int
    sixes: int
    fours: int
    
    sr: float
    bpb: float
    boundary_percent: float
    summary: str

In [20]:
def calculate_sr(state: BatsmanState):
    sr = (state['runs']/state['balls'])*100
    state['sr'] = sr
    return {'sr': sr}
    

In [21]:
def calculate_bpb(state: BatsmanState):
    bpb = state['balls']/(state['fours']+state['sixes'])
    state['bpb'] = bpb
    return {'bpb': bpb}

In [22]:
def calculate_boundary_percent(state: BatsmanState):
    boundary_percent = ((state['fours']*4 + state['sixes']*6)/state['runs'])*100
    state['boundary_percent'] = boundary_percent
    return {'boundary_percent': boundary_percent}

In [23]:
def summary(state: BatsmanState):
    summary = f"""
    Strike Rate - {state['sr']}\n
    Balls per Boundary - {state['bpb']} \n
    Boundary Percent - {state['boundary_percent']}
    """
    state['summary'] = summary
    
    return {'summary' : summary}

In [24]:
graph= StateGraph(BatsmanState)
# Nodes
graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percent', calculate_boundary_percent)
graph.add_node('summary', summary)
#Edges
graph.add_edge(START, 'calculate_sr' )
graph.add_edge(START, 'calculate_bpb' )
graph.add_edge(START, 'calculate_boundary_percent' )

graph.add_edge('calculate_sr', 'summary' )
graph.add_edge('calculate_bpb' , 'summary' )
graph.add_edge('calculate_boundary_percent', 'summary'  )

graph.add_edge('summary', END)

workflow = graph.compile()

In [27]:
initial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 5
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'runs': 100, 'balls': 50, 'sixes': 5, 'fours': 6, 'sr': 200.0, 'bpb': 4.545454545454546, 'boundary_percent': 54.0, 'summary': '\n    Strike Rate - 200.0\n\n    Balls per Boundary - 4.545454545454546 \n\n    Boundary Percent - 54.0\n    '}


In [28]:
print(final_state['summary'])


    Strike Rate - 200.0

    Balls per Boundary - 4.545454545454546 

    Boundary Percent - 54.0
    
